# Resumen de la notebook transformers from scratch

## Notas generales



#### El transformer en modo inferencia (generativo)
* El transformer es un modelo generativo que genera texto. Recibe como entrada una secuencia de $n$ tokens correspondientes a un vocabulario y responde a la salida con una distribución de probabilidad sobre todos los posibles tokens de ese vocabulario.
* Al muestrear dicha distribución de probabilidad se obtiene el token $n+1$ que corresponde al token que con mayor probabilidad el transformer cree que seguiría a la secuencia de los $n$ tokens de entrada. Luego se genera una nueva secuencia de entrada formada por los $n+1$ tokens y se obtiene el token $n+2$ y así sucesivamente. 

#### El transformer en modo entrenamiento
El modelo se entrena con secuencias de $n$ tokens de entrenamiento. Cada secuencia $(x_1,\ldots,x_{n-1},x_n)$ es aplicada a la entrada. El modelo genera $n$ distribuciones de probabilidad sobre los tokens del vocabulario. Cada distribución corresponde a la probabilidad del siguiente token dada la historia de los prefijos anteriores, es decir, se obtiene:

$$
p(x_1), \; p(x_2|x_1), \; p(x_3|x_1x_2), \; \ldots, \; p(x_n|x_1 \ldots x_{n-1})
$$


<details>
<summary> Transformer overview </summary>

<img src="./transformer-overview.png" width="900">

</details>


In [1]:
import transformer_lens
import math
import os
import sys
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import datasets
import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast

device = t.device(
    "cuda:1" if t.cuda.device_count() >= 2 
    else "cuda:0" if t.cuda.device_count() == 1 
    else "cpu"
)

print(device)

cuda:0


## Implementación del transformer

### Parámetros del modelo

In [2]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12


cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


### Bloques auxiliares: Layer normalization, embedding y positional embedding y unenbedding

<details>
<summary> Layer normalization </summary>

<img src="./transformer-ln.png" width="900">

</details>

In [3]:
#--------------------------Layer Normalization----------------------------------
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        resmed = residual.mean(dim=-1, keepdim=True)
        resvar = t.sqrt(residual.var(dim=-1, keepdim=True,unbiased=False)+self.cfg.layer_norm_eps)
        au = (residual - resmed)/resvar*self.w +self.b
        return au

#--------------------------Embedding---------------------------------------------
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        return self.W_E[tokens]


#---------------------------Positional embedding----------------------------------
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        batch, ctx = tokens.shape
        return einops.repeat(self.W_pos[:ctx], 'ctx d_m -> b ctx d_m', b=batch)

#---------------------------Unenbedding----------------------------------------------
class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(t.zeros((cfg.d_vocab)), requires_grad=False)

    def forward(
        self, normalized_resid_final: Float[Tensor, "batch position d_model"]
    ) -> Float[Tensor, "batch position d_vocab"]:
        e = einops.einsum(normalized_resid_final, self.W_U, 'b q dm, dm e -> b q e') +self.b_U
        return e

### Bloque de atención

<details>
<summary> Attention block</summary>

<img src="./attention.png" width="900">

</details>


In [4]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def forward(self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        # Paso 1: determino Q, K y V, es decir proyectamos d_model sobre d_head
        Q = einops.einsum(normalized_resid_pre, self.W_Q,
            'batch posn d_model, n_heads d_model d_heads -> batch posn n_heads d_heads') + self.b_Q
        K = einops.einsum(normalized_resid_pre, self.W_K,
            'batch posn d_model, n_heads d_model d_heads -> batch posn n_heads d_heads') + self.b_K
        V = einops.einsum(normalized_resid_pre, self.W_V,
            'batch posn d_model, n_heads d_model d_heads -> batch posn n_heads d_heads') + self.b_V
        # Paso 2: hacemos QK^T
        att_score = einops.einsum(Q,K, 
        'batch posn_q n_heads d_heads, batch posn_k n_heads d_heads -> batch n_heads posn_q posn_k')
        # Paso 3:  Escalamos
        att_score /= self.cfg.d_head**0.5
        # Paso 4: Aplicamos la máscara
        att_score = self.apply_causal_mask(att_score)
        # Paso 5: Convertimos el score en probabilidad con softmax
        A = att_score.softmax(dim=-1)
        # Paso 6: Hallamos los vectores ponderados z
        Z = einops.einsum(A, V, 'b nh q k, b k nh dh -> b q nh dh')
        # Paso 7: Obtenemos O, es decir proyectamos d_head sobre d_model
        O = einops.einsum(self.W_O, Z, 'nh dh dm, b q nh dh -> b q dm') + self.b_O
        return O

    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        # Define a mask that is True for all positions we want to set probabilities to zero for
        mask = t.ones(attn_scores.shape[-2], attn_scores.shape[-1],device=attn_scores.device)
        mask = t.triu(mask,diagonal=1).bool()

        # Apply the mask to attention scores, then return the masked scores
        return attn_scores.masked_fill(mask,self.IGNORE)
        



### Bloque de perceptrón multicapa

<details>
<summary> MLP block </summary>

<img src="./mlp-block.png" width="900">

</details>



In [5]:
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
        self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
        self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
        self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)

    def forward(self, normalized_resid_mid: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        z = einops.einsum(normalized_resid_mid, self.W_in, 'b q dm, dm dmlp -> b q dmlp') + self.b_in
        y = gelu_new(z)
        o = einops.einsum(y, self.W_out, 'b q dmlp, dmlp dm -> b q dm') + self.b_out
        return o



### Bloque transformer

<details>
<summary> Transformer block </summary>

<img src="./transformer-block.png" width="900">

</details>

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(self, resid_pre: Float[Tensor, "batch position d_model"]) -> Float[Tensor, "batch position d_model"]:
        o1 = self.ln1(resid_pre)
        o2 = self.attn(o1)
        o3 = o2 + resid_pre
        o4 = self.ln2(o3)
        o5 = self.mlp(o4)
        o6 = o3 + o5
        return o6

### Transformer

<details>
<summary> Transformer </summary>

<img src="./transformer.png" width="900">

</details>

In [7]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
        en = self.embed(tokens)
        posen = self.pos_embed(tokens)
        x = en + posen
        for blo in self.blocks:
            x = blo(x)
        y = self.ln_final(x)
        unen = self.unembed(y)
        return unen


## Testeo del transformer

In [ ]:
tokens = reference_gpt2.to_tokens(reference_text).to(device)
string = reference_gpt2.to_str_tokens(tokens)